# 10 · Validation Gate

**What this notebook covers**

- Global checks: raw data hash, artifact chain, no duplicates across splits, stratification, test isolation
- Per-model checks: no leakage, correct feature set, preprocessing fitted on Train only, CV setup, data files read, artifact hash
- Check matrix plot and list of failures and warnings
- Save the gate report used by notebooks 11 to 14

## 1. Environment setup
### 1.1 Imports

In [ ]:
import hashlib
import json
import platform
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

### 1.2 Configuration, seed and paths

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_PREFIX = "10"
DOCUMENTED_RAW_SHA256 = "7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06"
MODEL_NOTEBOOK_PREFIXES = ("05", "06", "07", "08", "09")
MAX_STRATIFICATION_GAP_PP = 0.5
MIN_CV_FOLDS = 5
# Static patterns that would indicate a model notebook touched the held-out split.
TEST_REFERENCE_PATTERN = re.compile(r"X_test|y_test|test\.csv|[\"']test[\"']")

# Resolve the repository root regardless of where the kernel was started
# (VS Code starts kernels in notebooks/, Jupyter may start them at the repo root).
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "DECISIONS.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (no DECISIONS.md in any parent directory).")


PROJECT_ROOT = find_project_root()

PATHS = {
    "raw": PROJECT_ROOT / "data" / "hotel_bookings.csv",
    "clean": PROJECT_ROOT / "data" / "interim" / "hotel_bookings_clean.csv",
    "splits": PROJECT_ROOT / "data" / "processed" / "splits",
    "features": PROJECT_ROOT / "data" / "processed" / "features",
    "artifacts": PROJECT_ROOT / "artifacts",
    "preprocessors": PROJECT_ROOT / "artifacts" / "preprocessors",
    "model_outputs": PROJECT_ROOT / "artifacts" / "model_outputs",
    "models": PROJECT_ROOT / "models",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "tables": PROJECT_ROOT / "reports" / "tables",
    "notebooks": PROJECT_ROOT / "notebooks",
}
for key in ("artifacts", "preprocessors", "model_outputs", "models", "figures", "tables"):
    PATHS[key].mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

### 1.3 Shared constants and helpers

In [ ]:
# Columns that must always be treated as categorical strings when re-reading CSVs.
STRING_COLUMNS = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "agent", "company",
    "customer_type", "arrival_season",
]
TARGET = "is_canceled"
ID_COLUMN = "booking_id"
LEAKAGE_COLUMNS = ["reservation_status", "reservation_status_date"]

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Streaming SHA-256 of a file (used to fingerprint every artifact)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path.relative_to(PROJECT_ROOT)}")
    return json.loads(path.read_text())


def read_frame(path: Path) -> pd.DataFrame:
    """Read a project CSV with categorical columns forced to plain Python strings.

    `keep_default_na=False` + an explicit NA list stops pandas from turning legitimate
    category labels into NaN; numeric columns are parsed normally.
    """
    if not path.exists():
        raise FileNotFoundError(f"Required data file missing: {path.relative_to(PROJECT_ROOT)}")
    frame = pd.read_csv(
        path,
        dtype={col: object for col in STRING_COLUMNS},
        keep_default_na=False,
        na_values=["", "NaN", "nan"],
    )
    present = [c for c in STRING_COLUMNS if c in frame.columns]
    frame[present] = frame[present].astype(object)
    return frame

In [ ]:
import importlib.metadata as importlib_metadata

def library_versions(packages=("numpy", "pandas", "scikit-learn", "xgboost", "matplotlib", "seaborn", "scipy", "shap", "joblib")) -> dict:
    versions = {"python": sys.version.split()[0], "platform": platform.platform()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not installed"
    return versions

LIBRARY_VERSIONS = library_versions()
pd.Series(LIBRARY_VERSIONS, name="version").to_frame()

### 1.4 Check-recording helper
Every check produces a structured record (id, scope, status, detail). `FAIL` blocks
eligibility; `WARN` is reported but does not block, and needs a written justification.

In [ ]:
CHECKS: list[dict] = []


def record(check_id: str, scope: str, passed: bool, detail: str, warn_only: bool = False) -> bool:
    status = "PASS" if passed else ("WARN" if warn_only else "FAIL")
    CHECKS.append({"check": check_id, "scope": scope, "status": status, "detail": detail})
    return passed

## 2. Load upstream contracts

In [ ]:
data_contract = load_json(PATHS["artifacts"] / "data_contract.json")
split_manifest = load_json(PATHS["artifacts"] / "split_manifest.json")
feature_spec = load_json(PATHS["artifacts"] / "feature_spec.json")

card_paths = sorted(PATHS["model_outputs"].glob("*/model_card.json"))
model_cards = {path.parent.name: load_json(path) for path in card_paths}
assert model_cards, "No model cards found - run notebooks 05-09 first."
print(f"Model cards found ({len(model_cards)}): {list(model_cards)}")

## 3. Global checks
### G1: Raw data fingerprint

In [ ]:
raw_hash = sha256_file(PATHS["raw"])
record("G1", "global", raw_hash == DOCUMENTED_RAW_SHA256 == data_contract["raw"]["sha256"],
       f"raw sha256 {raw_hash[:16]}...")

### G2: Artifact chain integrity
Each artifact on disk must match the hash recorded by the notebook that consumed it.
A mismatch means a file was regenerated without re-running its downstream notebooks.

In [ ]:
chain = [
    ("clean file vs data_contract", sha256_file(PATHS["clean"]), data_contract["clean"]["sha256"]),
    ("split manifest source vs data_contract", split_manifest["source"]["sha256"], data_contract["clean"]["sha256"]),
    *[(f"{name} split vs manifest", sha256_file(PATHS["splits"] / f"{name}.csv"), split_manifest["file_hashes"][name])
      for name in ("train", "dev", "test")],
    *[(f"feature spec source[{name}] vs manifest", feature_spec["source_split_hashes"][name], split_manifest["file_hashes"][name])
      for name in ("train", "dev", "test")],
    *[(f"{name} features vs feature_spec", sha256_file(PATHS["features"] / f"{name}.csv"), feature_spec["file_hashes"][name])
      for name in ("train", "dev", "test")],
]
broken = [label for label, actual, expected in chain if actual != expected]
record("G2", "global", not broken, "all links intact" if not broken else f"broken: {broken}")

### G3: De-duplication happened before the split
IDs must be disjoint, and no *content* may appear in two splits (hash of every column
except the ID). This is the executable form of decision #4.

In [ ]:
splits = {name: read_frame(PATHS["splits"] / f"{name}.csv") for name in ("train", "dev", "test")}
clean = read_frame(PATHS["clean"])

content_cols = [c for c in clean.columns if c != ID_COLUMN]
hashes = {name: set(pd.util.hash_pandas_object(frame[content_cols], index=False)) for name, frame in splits.items()}
ids = {name: set(frame[ID_COLUMN]) for name, frame in splits.items()}
pairs = [("train", "dev"), ("train", "test"), ("dev", "test")]

content_overlap = {f"{a}-{b}": len(hashes[a] & hashes[b]) for a, b in pairs}
id_overlap = {f"{a}-{b}": len(ids[a] & ids[b]) for a, b in pairs}
union_ok = set().union(*ids.values()) == set(clean[ID_COLUMN])
no_dups_in_clean = not clean[content_cols].duplicated().any()
record("G3", "global", sum(content_overlap.values()) == 0 and sum(id_overlap.values()) == 0 and union_ok and no_dups_in_clean,
       f"content overlap {content_overlap}, id overlap {id_overlap}, union==clean {union_ok}, clean has no dups {no_dups_in_clean}")

### G4: Stratification preserved

In [ ]:
overall = clean[TARGET].mean()
gaps = {name: abs(frame[TARGET].mean() - overall) * 100 for name, frame in splits.items()}
record("G4", "global", max(gaps.values()) <= MAX_STRATIFICATION_GAP_PP,
       "gaps (pp): " + ", ".join(f"{k}={v:.3f}" for k, v in gaps.items()))

### G5: Test isolation (static source scan of model notebooks)
Code cells of notebooks 05–09 are scanned for any reference to the held-out split. A
match means a model notebook could have used Test during development, which would make
the final Test numbers optimistic.

In [ ]:
def code_cells(notebook_path: Path) -> list[str]:
    notebook = json.loads(notebook_path.read_text())
    return ["".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code"]


scan_hits = {}
model_notebooks = sorted(p for p in PATHS["notebooks"].glob("*.ipynb") if p.name.startswith(MODEL_NOTEBOOK_PREFIXES))
for notebook_path in model_notebooks:
    hits = sorted({m.group() for src in code_cells(notebook_path) for m in TEST_REFERENCE_PATTERN.finditer(src)})
    if hits:
        scan_hits[notebook_path.name] = hits
record("G5", "global", not scan_hits and len(model_notebooks) > 0,
       f"scanned {len(model_notebooks)} notebooks; hits: {scan_hits or 'none'}")

## 4. Per-model checks
### 4.1 Helpers to introspect fitted pipelines
- `consumed_columns` reads the columns the **fitted** `ColumnTransformer` actually routes to
  its transformers. This is the ground truth, not the input frame's columns.
- `learned_state` extracts every learned statistic (imputer medians, encoder categories,
  scaler means and scales, row counts) so it can be compared with a fresh Train-only fit.

In [ ]:
def get_preprocessor(pipeline) -> ColumnTransformer:
    assert isinstance(pipeline, Pipeline) and "preprocess" in pipeline.named_steps, "Model is not a Pipeline with a 'preprocess' step"
    return pipeline.named_steps["preprocess"]


def consumed_columns(preprocessor: ColumnTransformer) -> set[str]:
    columns = set()
    for name, _, cols in preprocessor.transformers_:
        if name != "remainder" and not isinstance(cols, slice):
            columns.update(cols)
    return columns


def learned_state(preprocessor: ColumnTransformer) -> dict:
    state = {}
    for name, transformer in preprocessor.named_transformers_.items():
        if name == "remainder":
            continue
        steps = transformer.named_steps if isinstance(transformer, Pipeline) else {name: transformer}
        for step_name, step in steps.items():
            for attr in ("statistics_", "mean_", "scale_", "n_samples_seen_"):
                if hasattr(step, attr):
                    state[f"{name}.{step_name}.{attr}"] = np.asarray(getattr(step, attr))
            if hasattr(step, "categories_"):
                for i, cats in enumerate(step.categories_):
                    state[f"{name}.{step_name}.categories_[{i}]"] = np.asarray(cats, dtype=object)
            if hasattr(step, "infrequent_categories_"):
                for i, cats in enumerate(step.infrequent_categories_):
                    state[f"{name}.{step_name}.infrequent_[{i}]"] = np.asarray([] if cats is None else cats, dtype=object)
    return state


def states_equal(a: dict, b: dict) -> tuple[bool, list[str]]:
    differences = [k for k in sorted(set(a) | set(b)) if k not in a or k not in b
                   or a[k].shape != b[k].shape
                   or not (np.allclose(a[k].astype(float), b[k].astype(float)) if a[k].dtype != object else (a[k] == b[k]).all())]
    return not differences, differences

### 4.2 Load Train / Dev features once (inputs for the fit-on-train-only test)

In [ ]:
train_features = read_frame(PATHS["features"] / "train.csv")
X_train_ref = train_features.drop(columns=[TARGET, ID_COLUMN])
y_train_ref = train_features[TARGET].astype(int)
n_train = len(X_train_ref)

POST_BOOKING = set(feature_spec["post_booking_columns"])
FORBIDDEN = set(feature_spec["leakage_columns"]) | {TARGET, ID_COLUMN}
SPEC_COLUMNS = {variant: {c for cols in spec.values() for c in cols} for variant, spec in feature_spec["variants"].items()}

### 4.3 Run checks M1–M7 for every model
For M3, the gate refits a **fresh clone** of each model's preprocessor on Train and
asserts the learned state is identical. Any statistic learned on more (or other) rows
would differ. `n_samples_seen_` must equal the Train row count wherever it exists.

In [ ]:
for model_id, card in model_cards.items():
    pipeline = joblib.load(PROJECT_ROOT / card["model_path"])
    preprocessor = get_preprocessor(pipeline)
    used = consumed_columns(preprocessor)
    variant = card["variant"]
    reference_variant = "day_zero" if variant == "baseline" else variant

    # M1 - no leakage / target among consumed columns
    record("M1", model_id, not used & FORBIDDEN, f"forbidden columns consumed: {sorted(used & FORBIDDEN) or 'none'}")

    # M2 - variant integrity
    matches_spec = used == SPEC_COLUMNS[reference_variant]
    post_booking_ok = variant == "full_lifecycle" or not used & POST_BOOKING
    record("M2", model_id, matches_spec and post_booking_ok,
           f"variant={variant}; equals spec: {matches_spec}; post-booking used: {sorted(used & POST_BOOKING) or 'none'}")

    # M3 - fitted on Train only
    fresh = clone(preprocessor).fit(X_train_ref, y_train_ref)
    same, diffs = states_equal(learned_state(preprocessor), learned_state(fresh))
    seen_counts = {k: int(v) for k, v in learned_state(preprocessor).items() if k.endswith("n_samples_seen_")}
    counts_ok = all(v == n_train for v in seen_counts.values())
    record("M3", model_id, same and counts_ok,
           f"state identical to Train-only refit: {same}{'' if same else f' (diffs: {diffs[:3]})'}; n_samples_seen ok: {counts_ok}")

    # M4 - CV configuration
    cv = card["cv"]
    cv_ok = cv["type"] == "StratifiedKFold" and cv["n_splits"] >= MIN_CV_FOLDS and cv["shuffle"] and cv["random_state"] == RANDOM_STATE
    record("M4", model_id, cv_ok, f"{cv['type']} k={cv['n_splits']} shuffle={cv['shuffle']} seed={cv['random_state']}")
    if cv["n_rows"] != n_train:
        record("M4b", model_id, False, f"CV ran on {cv['n_rows']:,} of {n_train:,} Train rows (subsample; documented in card)", warn_only=True)

    # M5 - only Train/Dev files read
    files = card["data_files_read"]
    record("M5", model_id, all(Path(f).name in {"train.csv", "dev.csv"} for f in files) and len(files) > 0, f"files read: {files}")

    # M6 - weighting, no resampling
    record("M6", model_id, card["n_fit_rows"] == n_train,
           f"fit rows {card['n_fit_rows']:,} vs Train {n_train:,}; strategy: {card['imbalance_strategy']}")

    # M7 - artifact integrity
    record("M7", model_id, sha256_file(PROJECT_ROOT / card["model_path"]) == card["model_sha256"], "model file hash vs card")

print(f"{len(CHECKS)} checks recorded.")

## 5. Results
### 5.1 Global checks

In [ ]:
checks = pd.DataFrame(CHECKS)
checks[checks["scope"] == "global"]

### 5.2 Per-model check matrix
One row per model, one column per check. Any `FAIL` makes the model ineligible.

In [ ]:
per_model = checks[checks["scope"] != "global"]
matrix = per_model.pivot_table(index="scope", columns="check", values="status", aggfunc="first").fillna("-")
global_ok = (checks.loc[checks["scope"] == "global", "status"] != "FAIL").all()
matrix["eligible"] = matrix.apply(lambda row: global_ok and "FAIL" not in row.values, axis=1)
matrix

### 5.3 Failures and warnings in detail

In [ ]:
issues = checks[checks["status"] != "PASS"]
issues if len(issues) else "No failures or warnings."

## 6. Persist the gate report
Notebooks 11–14 read `eligible_models` from this file. Ineligible models are excluded
automatically and listed with their failure reasons.

In [ ]:
checks.to_csv(PATHS["tables"] / "10_validation_gate_checks.csv", index=False)
matrix.to_csv(PATHS["tables"] / "10_validation_gate_matrix.csv")

gate_report = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "10_validation_gate.ipynb",
    "global_passed": bool(global_ok),
    "eligible_models": sorted(matrix.index[matrix["eligible"]].tolist()),
    "ineligible_models": {m: per_model.loc[(per_model["scope"] == m) & (per_model["status"] == "FAIL"), "detail"].tolist()
                          for m in matrix.index[~matrix["eligible"]]},
    "warnings": issues[issues["status"] == "WARN"].to_dict(orient="records"),
    "checks": CHECKS,
    "model_card_hashes": {m: sha256_file(p) for m, p in zip(model_cards, card_paths)},
    "library_versions": LIBRARY_VERSIONS,
}
save_json(gate_report, PATHS["artifacts"] / "validation_gate_report.json")
print(f"Global checks passed: {global_ok}")
print(f"Eligible models ({len(gate_report['eligible_models'])}): {gate_report['eligible_models']}")
if gate_report["ineligible_models"]:
    print("INELIGIBLE:", json.dumps(gate_report["ineligible_models"], indent=2))

## 7. Sign-off
- [ ] All global checks `PASS`.
- [ ] Every `WARN` has a written justification (e.g. SVM subsample CV, decision #18 / notebook 09).
- [ ] Any `FAIL` has been fixed in its notebook and this gate re-run. Never edit the gate to pass.
- [ ] Record the gate outcome in `PROGRESS.md` ("Validation gate" row).